# ⚖️ Tüketici Kanunu PDF RAG Asistanı (End-to-End PDF RAG Project)

Bu projede, resmi Tüketici Kanunu PDF mevzuat belgesi üzerinde **semantik arama** ve **soru-cevap asistanı** inşa edilmektedir.

---

### 🛠️ Sistem Mimarisi:
1. **Belge Yükleme:** `PyPDFLoader` ile PDF sayfalarının ayrıştırılması.
2. **Metin Parçalama:** `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)`.
3. **Vektörleştirme & İndeksleme:** Dense Embeddings ile vektör uzayına aktarım ve ChromaDB vektör deposu.
4. **Geri Getirme & Yanıtlama (RAG Chain):** Kullanıcı sorusuna en uygun kanun maddelerinin getirilmesi ve LLM ile gerekçeli hukuki yanıt üretimi.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os

In [ ]:
sayfalar = PyPDFLoader("buyuk-veri/tuketici-kanunu.pdf").load()
print(f"{len(sayfalar)} sayfa yüklendi")
print("Örnek:", " ".join(sayfalar[0].page_content.split())[:150])

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 100)
chunks = splitter.split_documents(sayfalar)
print(f"{len(sayfalar)} sayfa -> {len(chunks)} chunk")

print("metadata örneği:", chunks[50].metadata)

In [ ]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-samll")

vectordb = Chroma.from_documents(
    chunks,
    embeddings = embeddings,
    persist_directory = "./kanun_db_v2",
    collection_name = "tuketici_kanunu"
)
print(vectordb._colleciton.count(), "chunk indekslendi -> ./kanun_db klasörüne kaydedildi")

In [ ]:
for d in vectordb.similarity_search("cayma hakkı kaç gündü?", k = 3):
    print(f"[sayfa {d.metadata["page"]+1}]. {" ".join(d.page_content.split())[:90]}...")

In [ ]:
llm = ChatOpenAI(model = "gpt-5.4-mini", temperature = 0)
retriever = vectordb.as_retriever(search_kwargs = {"k": 4})

PROMPT = """
Sen tüketici hakları yardımcı bir asistansın. SADECE aşağıdaki kanun metni alınır 
Cevap alıntılarda yoksa "Bu konuda kanun metninde bilgi bulamadım" de. sade Türkçeyle, kısa açıkla.

KANUN METNİ:
{baglam}

SORU: {soru}
CEVAP: """

def sor(soru, k = 4):
    docs = retriever.invoke(soru)[:k]
    baglam = "\n\n".join(d.page_content for d in docs)
    cevap = llm.invoke(PROMPT.format(baglam = baglam, soru = soru)).content
    sayfalar_k = sorted({d.metadata["page"] + 1 for d in docs})
    return cevap, sayfalar

cevap, kaynak = sor("İnternetten aldığım ürünü kaç gün içinde iade edebilirim?")
print(cevap)
print("\n Kaynak: kanun sayfa", ", ".join(map(str, kaynak)))

In [ ]:
for s in ["Ayıplı mal nedir? ne yapabilirim?",
          "Granti süresi en az kaç yıldır?",
          "Sepetix'in üyelik süreci ne kadar?"]:
    c,k = sor(s)
    print(f"S: {s}\n{c}\n. sayfa. {", ".join(map(str,k))}\n{"-"*60}")

In [ ]:
gold = [
    ("cayma hakkı kaç gündür?",                         "on dört gün"),
    ("ayıplı mal nedir?"                                "ayıplı"),
    ("üretici garanti belgesi düzenlemek zorunda mı?", "garanti belgesi"),
    ("abonelik sözleşmesi nasıl feshedilir?"           "fesih"),
    ("taksitle satışta erken ödeme yapılabilir mi?"    "erken ödeme")
]
hit = 0
for soru, beklenen in gold:
    getirilen = " ".join(d.page_conetnt.lower() for d in vectordb.similarity_search(soru, k = 4))
    ok = beklenen in getirilen
    hit += ok
    print(f"{':)' if ok else "x"}   {soru} -> {beklenen}    {"bulundu" if ok else "YOK"}")

print(f"\nHit-rate@4: {hit}/{len(gold)}")

# Özet

- 53 SAYFA RESMİ pdf -> PyPDFLoader ile yükle

- chunk-size veriye göre: 400 -> 800

- persist_directory ile kalıcı Chroma DB

- Sayfa numaralı kaynak gösteren asistan

- Gold set + hit-rate ile ölçüm